In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torchvision.datasets import MNIST, CIFAR10, CIFAR100, STL10
from tqdm.auto import tqdm
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import shapiro, normaltest
import importlib
import pandas as pd
import skimage.io as io
from math import log, sqrt
from mhnlib.fixed_points import get_symmetric_stability_matrix_gram, get_entropies, get_jacobian_gram, get_symmetric_stability_matrices_gram
from mhnlib.dynamics import DualDeterministicDynamics, Dynamics

In [ ]:
%load_ext autoreload

### Helper functions

In [ ]:
class PokemonDataset(Dataset):
    def __init__(
        self,
        root = "",
        variant="normal",
        game="red-blue",
        image_col="local_path",   # "local_path" = color, "bw_path" = black/white
        size=32,
        one_per_pokemon=True,
        mode="rgb",               # "rgb", "rgba", or "mask"
    ):
        self.root = root
        info = pd.read_csv(self.root + "pokemondb_sprites/metadata.csv")

        sub_info = info[
            (info["game"] == game)
            & (info["variant"] == variant)
        ].copy()

        sub_info = sub_info[
            sub_info[image_col].notna()
            & (sub_info[image_col].astype(str) != "")
        ]

        if one_per_pokemon:
            sub_info = sub_info.groupby("pokemon", as_index=False).first()

        self.paths = sub_info[image_col].astype(str).tolist()
        self.names = sub_info["pokemon"].astype(str).tolist()

        self.classes = sorted(set(self.names))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.labels = [self.class_to_idx[n] for n in self.names]

        self.size = size
        self.mode = mode

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.root + self.paths[idx]
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        img = Image.open(path).convert("RGBA")

        if self.size is not None:
            img = img.resize((self.size, self.size), Image.Resampling.NEAREST)

        arr = np.asarray(img).astype(np.float32) / 255.0

        if self.mode == "mask":
            # Binary alpha mask: [1, H, W]
            alpha = arr[..., 3]
            x = torch.from_numpy((alpha > 0).astype(np.float32)).unsqueeze(0)

        elif self.mode == "rgba":
            # Keep transparency: [4, H, W]
            x = torch.from_numpy(arr).permute(2, 0, 1)

        elif self.mode == "rgb":
            # Composite transparent background onto white: [3, H, W]
            rgb = arr[..., :3]
            alpha = arr[..., 3:4]
            bg = np.ones_like(rgb)
            rgb = alpha * rgb + (1.0 - alpha) * bg
            x = torch.from_numpy(rgb).permute(2, 0, 1)

        else:
            raise ValueError(f"Unknown mode: {self.mode}")

        return x, label

In [ ]:
def shift_and_rms(x, eps=1e-12):
    shift = x.mean(dim=0, keepdim=True)
    x_shifted = x - shift

    rms = x_shifted.norm(dim=1).square().mean().sqrt()
    rms = rms.clamp_min(eps)

    x_norm = x_shifted / rms

    return shift, rms, x_norm

In [ ]:
@torch.no_grad()
def decode_latents(z, z_mean, z_std, vae_model, vae_scaling, batch_size=0, verbose=False):
    orig_ndim = z.ndim

    if z.ndim == 4:
        B, C, H, W = z.shape
        vae_input = z * z_std + z_mean

    elif z.ndim == 5:
        B1, B2, C, H, W = z.shape
        vae_input = z.reshape(B1 * B2, C, H, W)
        vae_input = vae_input * z_std + z_mean

    else:
        raise ValueError("z must have 4 or 5 dimensions.")

    vae_input = vae_input / vae_scaling

    if batch_size > 0:
        vae_decoded = []

        chunks = torch.split(vae_input, batch_size, dim=0)

        for chunk in tqdm(chunks, disable=not verbose):
            chunk = chunk.to(vae_model.device)
            decoded = vae_model.decode(chunk).sample.cpu()
            vae_decoded.append(decoded)

        vae_decoded = torch.cat(vae_decoded, dim=0)

    else:
        vae_decoded = vae_model.decode(
            vae_input.to(vae_model.device)
        ).sample.cpu()

    if orig_ndim == 5:
        vae_decoded = vae_decoded.view(B1, B2, *vae_decoded.shape[1:])

    return (1 + vae_decoded.clamp(-1, 1)) / 2

In [ ]:
def play_images(
    images,
    time_variable,
    time_variable_name,
    num_cols,
    use_grayscale,
    base_image_size=5,
    pause=0.1,
):
    import time as time_module
    import numpy as np
    import matplotlib.pyplot as plt
    from IPython.display import display
    from skimage.color import rgb2gray

    num_runs = images.shape[0]
    num_times = images.shape[1]

    if num_cols is None:
        num_cols = num_runs

    num_rows = (num_runs + num_cols - 1) // num_cols

    fig, axs = plt.subplots(
        nrows=num_rows,
        ncols=num_cols,
        figsize=(base_image_size * num_cols, base_image_size * num_rows),
        squeeze=False,
    )

    axs = axs.flatten()
    ims = []

    def get_image(run, time_idx):
        img = images[run, time_idx]

        if hasattr(img, "detach"):
            img = img.detach().cpu()

        if img.shape[0] in (1, 3, 4):  # C, H, W
            img = img.permute(1, 2, 0)

        img = np.asarray(img)

        if img.shape[-1] == 1:
            img = img[..., 0]

        img = np.clip(img, 0, 1)

        if use_grayscale and img.ndim == 3:
            img = rgb2gray(img[..., :3])

        return img

    for run in range(num_runs):
        img = get_image(run, 0)

        if use_grayscale:
            im = axs[run].imshow(img, cmap="gray", vmin=0, vmax=1)
        else:
            im = axs[run].imshow(img)

        axs[run].axis("off")
        axs[run].set_title(f"run {run}")
        ims.append(im)

    for ax in axs[num_runs:]:
        ax.axis("off")

    fig.suptitle(
        f"Time step: {1}/{num_times}. "
        f"{time_variable_name}=${float(time_variable[0]):.2f}$",
        fontsize=20,
    )

    display_handle = display(fig, display_id=True)

    for time_idx in range(num_times):
        for run in range(num_runs):
            ims[run].set_data(get_image(run, time_idx))

        fig.suptitle(
            f"Time step: {time_idx + 1}/{num_times}. "
            f"{time_variable_name}=${float(time_variable[time_idx]):.2f}$",
            fontsize=20,
        )

        display_handle.update(fig)
        time_module.sleep(pause)

    plt.close(fig)
def animate_func(
    x,
    y,
    x0_vline,
    interval=100,
    repeat=True,
    fig_width=6,
    fig_height=4,
    save_path=None,
    x0_label=r"$x_0$",
    xscale="linear",   # "linear" or "log"
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation

    x = np.asarray(x)
    y = np.asarray(y)

    if x.ndim != 1 or y.ndim != 1:
        raise ValueError(
            f"x and y must be 1D arrays. Got x.shape={x.shape}, y.shape={y.shape}"
        )

    if x.shape[0] != y.shape[0]:
        raise ValueError(
            f"x and y must have same length. Got {x.shape[0]} and {y.shape[0]}"
        )

    if xscale not in ["linear", "log"]:
        raise ValueError(f"xscale must be 'linear' or 'log'. Got {xscale}")
    if x0_vline is not None:
        if xscale == "log" and np.any(x <= 0):
            raise ValueError("For xscale='log', all x values must be positive.")

        if xscale == "log" and x0_vline <= 0:
            raise ValueError("For xscale='log', x0_vline must be positive.")

    num_times = len(x)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    line, = ax.plot([], [], lw=2)

    ax.set_xscale(xscale)
    if x0_vline is not None:
        ax.axvline(
            x0_vline,
            ls="dashed",
            color="black",
            label=x0_label,
        )

        ax.text(
            x0_vline,
            1.02,
            x0_label,
            transform=ax.get_xaxis_transform(),
            ha="center",
            va="bottom",
        )

    xmin, xmax = np.nanmin(x), np.nanmax(x)
    ymin, ymax = np.nanmin(y), np.nanmax(y)

    if xscale == "log":
        ax.set_xlim(xmin / 1.05, xmax * 1.05)
    else:
        dx = 0.05 * (xmax - xmin + 1e-12)
        ax.set_xlim(xmin - dx, xmax + dx)

    dy = 0.05 * (ymax - ymin + 1e-12)
    ax.set_ylim(ymin - dy, ymax + dy)

    title = ax.set_title("")
    if x0_vline is not None:
        ax.legend()

    def update(t):
        line.set_data(x[: t + 1], y[: t + 1])
        title.set_text(f"frame = {t}")
        return line, title

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=num_times,
        interval=interval,
        blit=False,
        repeat=repeat,
    )

    if save_path is not None:
        fps = 1000 / interval
        if save_path.endswith(".gif"):
            anim.save(save_path, writer=animation.PillowWriter(fps=fps))
        else:
            anim.save(save_path, writer="ffmpeg", fps=fps)
    plt.close(fig)
    return anim
def animate_images(
    images,
    time_variable,
    time_variable_name,
    num_cols,
    use_grayscale,
    base_image_size=5,
    interval=100,
    repeat=True,
    save_path=None,
):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation
    from skimage.color import rgb2gray

    num_runs = images.shape[0]
    num_times = images.shape[1]
    num_rows = (num_runs + num_cols - 1) // num_cols

    fig, axs = plt.subplots(
        nrows=num_rows,
        ncols=num_cols,
        figsize=(base_image_size * num_cols, base_image_size * num_rows),
        squeeze=False,
    )

    axs = axs.flatten()
    ims = []

    # leave room for the suptitle
    fig.subplots_adjust(top=0.88)

    def get_image(run, time_idx):
        img = images[run, time_idx]

        if hasattr(img, "detach"):
            img = img.detach().cpu()

        if img.shape[0] in (1, 3, 4):  # C, H, W
            img = img.permute(1, 2, 0)

        img = np.asarray(img)

        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]

        img = np.clip(img, 0, 1)

        if use_grayscale and img.ndim == 3:
            img = rgb2gray(img[..., :3])

        return img

    for run in range(num_runs):
        img = get_image(run, 0)

        if use_grayscale:
            im = axs[run].imshow(img, cmap="gray", vmin=0, vmax=1)
        else:
            im = axs[run].imshow(img)

        axs[run].axis("off")
        axs[run].set_title(f"run {run}")
        ims.append(im)

    for ax in axs[num_runs:]:
        ax.axis("off")

    title = fig.suptitle(
        f"{time_variable_name}: {float(time_variable[0]):.4f} | frame: 0/{num_times-1}",
        fontsize=20,
    )

    def update(time_idx):
        for run in range(num_runs):
            ims[run].set_data(get_image(run, time_idx))

        t = float(time_variable[time_idx])
        title.set_text(
            f"{time_variable_name}: {t:.4f} | frame: {time_idx}/{num_times-1}"
        )

        return ims + [title]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=num_times,
        interval=interval,
        blit=False,   # important for suptitle visibility
        repeat=repeat,
    )

    if save_path is not None:
        if save_path.endswith(".gif"):
            anim.save(save_path, writer="pillow", dpi=120)
        else:
            anim.save(save_path, writer="ffmpeg", dpi=120)

    plt.close(fig)
    return anim

In [ ]:
torch.manual_seed(1101252)
IMG_SIZE = 32
DATASET = "cifar100"  # "cifar100", "stl10", "mnist", or "pokemon"

tfm = T.Compose([
    T.Resize(IMG_SIZE, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
])

if DATASET == "cifar100":
    dataset = CIFAR100(root="datasets/cifar100/", train=True, download=True, transform=tfm)
    test_set = CIFAR100(root="datasets/cifar100/", train=False, download=True, transform=tfm)

elif DATASET == "stl10":
    dataset = STL10(root="datasets/stl10/", split="train", download=True, transform=tfm)
    test_set = STL10(root="datasets/stl10/", split="test", download=True, transform=tfm)

elif DATASET == "mnist":
    dataset = MNIST(root="datasets/mnist/", train=True, download=True, transform=tfm)
    test_set = MNIST(root="datasets/mnist/", train=False, download=True, transform=tfm)
elif DATASET == "pokemon":
    dataset = PokemonDataset(root="datasets/", variant="normal", game="red-blue", image_col="local_path", size=IMG_SIZE, one_per_pokemon=True, mode="rgb")
    test_set = None
num_classes = len(dataset.classes)
print(f"Number of classes: {num_classes}")

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

In [ ]:
from diffusers import AutoencoderKL
ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
ae_model = ae_model.to(device).eval()
ae_model.requires_grad_(False)
ae_scaling = ae_model.config.scaling_factor

In [ ]:
torch.random.manual_seed(1101252)
MHN_BATCH_SIZE = 1024
mhn_data_loader = DataLoader(
    dataset,
    batch_size=MHN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)
data = []
latents = []
latents_labels = []
for (batch_x, batch_y) in tqdm(mhn_data_loader, desc="Encoding images with VAE"):
    batch_x = batch_x.to(device)
    with torch.no_grad():
        batch_x = batch_x.expand(-1, 3, -1, -1) if batch_x.shape[1] == 1 else batch_x
        z = ae_model.encode(batch_x).latent_dist.mode() * ae_scaling
    data.append(batch_x.cpu())
    latents.append(z.cpu())
    latents_labels.append(batch_y)
data = torch.cat(data, dim=0)
latents = torch.cat(latents, dim=0)
latents_labels = torch.cat(latents_labels, dim=0)

if test_set is not None:
    mhn_test_data_loader = DataLoader(
        test_set,
        batch_size=MHN_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    test_data = []
    test_latents = []
    test_latents_labels = []
    for (batch_x, batch_y) in tqdm(mhn_test_data_loader, desc="Encoding test images with VAE"):
        batch_x = batch_x.to(device)
        with torch.no_grad():
            batch_x = batch_x.expand(-1, 3, -1, -1) if batch_x.shape[1] == 1 else batch_x
            z = ae_model.encode(batch_x).latent_dist.mode() * ae_scaling
        test_data.append(batch_x.cpu())
        test_latents.append(z.cpu())
        test_latents_labels.append(batch_y)
    test_data = torch.cat(test_data, dim=0)
    test_latents = torch.cat(test_latents, dim=0)
    test_latents_labels = torch.cat(test_latents_labels, dim=0)

In [ ]:
C_latents, H_latents, W_latents = latents.shape[1:]

### Subsample latents

In [ ]:
# sample some latents according to label
torch.random.manual_seed(1101252)
center_and_scale_latents = True
samples_per_class = 100
sampled_data_indices = []
sampled_latents = []
sampled_labels = []
for c in range(num_classes):
    class_indices = (latents_labels == c).nonzero(as_tuple=True)[0]
    sampled_idxs= torch.randperm(len(class_indices))[:samples_per_class]
    sampled_latents.append(latents[class_indices[sampled_idxs]])
    sampled_labels.append(latents_labels[class_indices[sampled_idxs]])
    sampled_data_indices.append(class_indices[sampled_idxs])
sampled_latents = torch.cat(sampled_latents, dim=0)
sampled_labels = torch.cat(sampled_labels, dim=0)
sampled_data_indices = torch.cat(sampled_data_indices, dim=0)
if test_set is not None:
    samples_per_class_test = 1
    sampled_test_data = []
    sampled_test_data_indices = []
    sampled_test_latents = []
    sampled_test_labels = []
    for c in range(num_classes):
        class_indices = (test_latents_labels == c).nonzero(as_tuple=True)[0]
        sampled_idxs = torch.randperm(len(class_indices))[:samples_per_class_test]
        sampled_test_data_indices.append(class_indices[sampled_idxs])
        sampled_test_latents.append(test_latents[class_indices[sampled_idxs]])
        sampled_test_labels.append(test_latents_labels[class_indices[sampled_idxs]]) 
    sampled_test_latents = torch.cat(sampled_test_latents, dim=0)
    sampled_test_labels = torch.cat(sampled_test_labels, dim=0)
    sampled_test_data_indices = torch.cat(sampled_test_data_indices, dim=0)

if center_and_scale_latents:
    patterns_shift, patterns_rms, patterns = shift_and_rms(sampled_latents.reshape(sampled_latents.shape[0],-1))
else:
    patterns_shift = torch.zeros_like(sampled_latents[0:1]).reshape(1, -1)
    patterns_rms = torch.ones(1)
    patterns = torch.clone(sampled_latents.reshape(sampled_latents.shape[0], -1))

if test_set is not None:
    if center_and_scale_latents:
        patterns_test = (sampled_test_latents.reshape(sampled_test_latents.shape[0], -1) - patterns_shift) / patterns_rms
    else:
        patterns_test = sampled_test_latents.reshape(sampled_test_latents.shape[0], -1)

### Convergence of the test set:

In [ ]:
dynamics = Dynamics(patterns, biases=torch.zeros(patterns.shape[0]), is_stochastic=False)

In [ ]:
betas = torch.logspace(0, 3, steps=50)
test_retrieved_patterns = dynamics.integrate_fixed_point(patterns_test , betas, num_iterations=100, verbose=True)

In [ ]:
test_retrieved_images = decode_latents(
    test_retrieved_patterns.reshape(test_retrieved_patterns.shape[0], test_retrieved_patterns.shape[1], C_latents, H_latents, W_latents),
    z_mean=patterns_shift.reshape(C_latents, H_latents, W_latents),
    z_std=patterns_rms,
    vae_model=ae_model,
    vae_scaling=ae_scaling,
    batch_size=64,
    verbose=True,
)
#rgb2gra_vec= torch.tensor([0.2989, 0.5870, 0.1140], dtype=torch.float32)
#test_retrieved_images_gray = torch.einsum('...chw,c->...hw', test_retrieved_images, rgb2gra_vec)
#test_retrieved_images = test_retrieved_images_gray.unsqueeze(2)

In [ ]:
test_overlaps = torch.einsum('sbn,sn->sb', test_retrieved_patterns/torch.norm(test_retrieved_patterns, dim=1, keepdim=True), patterns_test/torch.norm(patterns_test, dim=1, keepdim=True))

In [ ]:
test_mse = torch.mean((test_retrieved_images- test_data[sampled_test_data_indices][:,None,...  ])**2, dim=(2, 3, 4))

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
def add_image_label(ax, x, y, img, zoom=0.35, xybox=(0, 35)):
    """
    Adds an image near point (x, y).

    xybox is the offset in screen points.
    """
    imagebox = OffsetImage(img, zoom=zoom)

    ab = AnnotationBbox(
        imagebox,
        (x, y),
        xybox=xybox,
        xycoords="data",
        boxcoords="offset points",
        frameon=True,
        arrowprops=dict(arrowstyle="->", lw=0.8),
        pad=0.2,
    )

    ax.add_artist(ab)
    return ab

In [ ]:
for idx in range(test_retrieved_images.shape[0]):
    fig, axs = plt.subplots(figsize=(10, 6), ncols=3)
    axs[2].plot(betas.cpu(), test_mse[idx].cpu())
    for beta_idx in range(0, len(betas), len(betas)//10):
        add_image_label(axs[2], betas[beta_idx].cpu(), test_mse[idx, beta_idx].cpu(), test_retrieved_images[idx, beta_idx].cpu().permute(1, 2, 0).numpy())
    axs[2].set_xscale("log")
    best_best_idx = torch.argmin(test_mse[idx])
    axs[0].imshow(test_data[sampled_test_data_indices[idx]].cpu().permute(1, 2, 0).numpy())
    axs[1].imshow(test_retrieved_images[idx, best_best_idx].cpu().permute(1, 2, 0).numpy())
    plt.show()

In [ ]:
for idx in range(test_retrieved_images.shape[0]):
    fig, axs = plt.subplots(ncols=len(betas) + 1, figsize=(10, 5))
    for i in range(len(betas)):
        axs[i].imshow(test_retrieved_images[idx, i].permute(1, 2, 0))
        axs[i].axis("off")
    axs[-1].imshow(test_data[sampled_test_data_indices[idx]].permute(1, 2, 0))
    axs[-1].axis("off")
    plt.show()